# Validation Run — Pipeline Dry-Run

**Goal:** Verify that `FloodDataset`, `standardized_rmse_loss`, and the
autoregressive inference loop work end-to-end *before* Member A/B deliver
trained models.

| Phase | Time-steps | Input |
|-------|-----------|-------|
| **Burn-in** | `t = 0 … BURN_IN-1` | Ground-truth (teacher forcing) |
| **Forecast** | `t = BURN_IN … T-1` | Previous prediction (autoregressive) |

In [ ]:
# ── Cell 1: Setup & Imports ───────────────────────────────────────────
# Auto-reload so edits in src/*.py are picked up without kernel restart
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from src.config import RAW_DATA_PATH
from src.dataset import FloodDataset
from src.loss import standardized_rmse_loss, standardized_rmse_metric

print(f"RAW_DATA_PATH = {RAW_DATA_PATH}")
print(f"PyTorch       = {torch.__version__}")
print(f"CUDA avail.   = {torch.cuda.is_available()}")

## Cell 2 — Data Loader Verification

Load the first event and check that static caching works.
Confirm that node counts are consistent between static and dynamic files.

In [ ]:
# ── Cell 2: Data Loader Verification ─────────────────────────────────
dataset = FloodDataset(root_dir=str(RAW_DATA_PATH), mode="train")
print(f"Total events in dataset: {len(dataset)}\n")

# Fetch the first event
event_0 = dataset[0]
print(f"Model ID : {event_0['model_id']}")
print(f"Event ID : {event_0['event_id']}")
print()

# ── Shape report ──────────────────────────────────────────────────────
for key in [
    "static_1d_nodes", "static_2d_nodes",
    "static_1d_edges", "static_2d_edges",
    "edge_index_1d",   "edge_index_2d",
    "1d2d_conn",
    "dynamic_1d_nodes", "dynamic_2d_nodes",
]:
    df = event_0[key]
    print(f"  {key:25s}  →  {df.shape}")

# ── Sanity check: node counts must agree ─────────────────────────────
# The dynamic CSV has one row per (timestep × node).  The first column
# is typically a node-id or index; the number of *unique* nodes should
# match the static file's row count.
n_static_1d  = len(event_0["static_1d_nodes"])
n_static_2d  = len(event_0["static_2d_nodes"])
print(f"\n  Static 1D nodes : {n_static_1d}")
print(f"  Static 2D nodes : {n_static_2d}")

# ── Cache verification ───────────────────────────────────────────────
# Fetching a second event from the *same* model should NOT re-read
# static files — they come from the cache.
if len(dataset) > 1:
    event_1 = dataset[1]
    if event_1["model_id"] == event_0["model_id"]:
        assert event_1["static_1d_nodes"] is event_0["static_1d_nodes"], \
            "Cache miss — static DataFrames should be the same object!"
        print("\n✓ Static cache verified (same object identity for same model).")
    else:
        print("\n⚠ Events 0 and 1 belong to different models; cache test skipped.")
else:
    print("\n⚠ Only one event in dataset; cache test skipped.")

print("\n✅ Data loader verification passed.")

## Cell 3 — Mock Model

A trivial `nn.Module` that returns its input plus small Gaussian noise.
This simulates a near-perfect predictor so we can test the full pipeline
without waiting for Member A/B's real models.

In [ ]:
# ── Cell 3: Mock Model ────────────────────────────────────────────────

class MockModel(nn.Module):
    """Identity + noise model for pipeline testing.

    Parameters
    ----------
    noise_std : float
        Standard deviation of additive Gaussian noise.
        Set to 0.0 for a perfect (identity) predictor.
    """

    def __init__(self, noise_std: float = 0.01):
        super().__init__()
        self.noise_std = noise_std

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor = None) -> torch.Tensor:
        """Return x + N(0, noise_std).

        ``edge_index`` is accepted (but ignored) so the call signature
        matches the real GCN / GraphSAGE models.
        """
        if self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

mock_model = MockModel(noise_std=0.02)
print(mock_model)

# Quick smoke test
dummy = torch.randn(5, 10)        # (T=5, N=10)
out   = mock_model(dummy)
assert out.shape == dummy.shape, "Shape mismatch!"
print(f"Smoke test passed — input {dummy.shape} → output {out.shape}")

## Cell 4 — Autoregressive Simulation

Simulates the **test-time inference loop** with two phases:

| Phase | Steps | Description |
|-------|-------|-------------|
| **Burn-in** | `0 … BURN_IN-1` | Feed ground-truth to the model (teacher forcing). Hidden states would be warmed up here in the real model. |
| **Forecast** | `BURN_IN … T-1` | Feed the *model's own output* from `t-1` as input for `t` (autoregressive). This is what gets scored. |

In [ ]:
# ── Cell 4: Autoregressive Simulation ─────────────────────────────────

# ── Hyperparameters ───────────────────────────────────────────────────
BURN_IN = 10   # Number of teacher-forced warm-up steps

# ── Extract ground-truth water levels as a tensor ─────────────────────
# The dynamic CSV typically has columns: [timestep, node_0, node_1, …]
# We drop the timestep column to get a pure (T, N) array.
dyn_df = event_0["dynamic_1d_nodes"]

# Identify numeric node columns (skip any non-numeric / id columns)
node_cols = [c for c in dyn_df.columns if c not in ("timestep", "time", "Timestep", "Time")]
gt_np = dyn_df[node_cols].values.astype(np.float32)  # (T, N)
gt = torch.from_numpy(gt_np)
T, N = gt.shape
print(f"Ground truth shape: T={T}, N={N}")
print(f"Burn-in steps     : {BURN_IN}")
print(f"Forecast steps    : {T - BURN_IN}")

# ── Build a dummy edge_index (fully disconnected — mock model ignores it)
edge_index = torch.zeros((2, 0), dtype=torch.long)

# ── Autoregressive loop ──────────────────────────────────────────────
predictions = []  # will collect forecast-phase predictions

mock_model.eval()
with torch.no_grad():
    for t in range(T):
        if t < BURN_IN:
            # ── BURN-IN PHASE ─────────────────────────────────────
            # Feed the *ground truth* at time t.
            # In a real GRU model this updates hidden states only;
            # we don't record predictions during burn-in.
            x_t = gt[t]                       # (N,)
            _ = mock_model(x_t.unsqueeze(0), edge_index)  # warm-up call
        else:
            # ── FORECAST PHASE ────────────────────────────────────
            # At the first forecast step, seed with the last GT frame.
            if t == BURN_IN:
                x_t = gt[t - 1]               # last burn-in frame
            # Predict the next step from the *previous prediction*
            pred_t = mock_model(x_t.unsqueeze(0), edge_index).squeeze(0)  # (N,)
            predictions.append(pred_t)
            # Autoregressive: output of t becomes input for t+1
            x_t = pred_t

# Stack predictions → (T_forecast, N)
preds_tensor = torch.stack(predictions, dim=0)
print(f"\nPredictions tensor : {preds_tensor.shape}  (should be ({T - BURN_IN}, {N}))")
assert preds_tensor.shape == (T - BURN_IN, N), "Shape mismatch!"
print("✅ Autoregressive loop completed successfully.")

## Cell 5 — Scoring

Compute the **Standardized RMSE** on the forecast window using both
the differentiable training loss and the exact leaderboard metric.

In [ ]:
# ── Cell 5: Scoring ───────────────────────────────────────────────────

# ── Ground-truth for the forecast window ─────────────────────────────
targets = gt[BURN_IN:]  # (T_forecast, N) — same window as predictions

# ── Per-node standard deviations ─────────────────────────────────────
# Computed over ALL time-steps (burn-in + forecast) to match the
# competition definition.  Shape: (N,)
node_stds = gt.std(dim=0)

# Show some stats about the std distribution
print("Per-node σ statistics:")
print(f"  min  = {node_stds.min().item():.6f}")
print(f"  mean = {node_stds.mean().item():.6f}")
print(f"  max  = {node_stds.max().item():.6f}")
n_near_zero = (node_stds < 1e-4).sum().item()
print(f"  nodes with σ ≈ 0 (< 1e-4): {n_near_zero} / {N}")

# ── Differentiable training loss ─────────────────────────────────────
loss_val = standardized_rmse_loss(preds_tensor, targets, node_stds)
print(f"\nStandardized RMSE Loss (training surrogate) : {loss_val.item():.6f}")
assert not torch.isnan(loss_val), "Loss is NaN!"
assert not torch.isinf(loss_val), "Loss is Inf!"

# ── Exact leaderboard metric ────────────────────────────────────────
srmse = standardized_rmse_metric(preds_tensor, targets, node_stds)
print(f"Standardized RMSE Metric (leaderboard)      : {srmse.item():.6f}")
assert not torch.isnan(srmse), "Metric is NaN!"

print("\n✅ Scoring verification passed — no NaNs, no Infs.")

## Cell 6 — Visualization

Plot ground truth vs. mock prediction for a single node to visually
confirm the autoregressive loop output makes sense.

In [ ]:
# ── Cell 6: Visualization ─────────────────────────────────────────────

# Pick a node to visualize (choose one with meaningful variance)
# Use the node with the highest σ so the plot is informative.
node_idx = int(node_stds.argmax().item())
node_label = node_cols[node_idx] if node_idx < len(node_cols) else f"Node {node_idx}"

# Full time axis
time_all = np.arange(T)
time_forecast = np.arange(BURN_IN, T)

# Ground-truth (all T steps) and predictions (forecast only)
gt_node = gt[:, node_idx].numpy()
pred_node = preds_tensor[:, node_idx].numpy()

fig, ax = plt.subplots(figsize=(12, 4))

# Ground truth — full series
ax.plot(time_all, gt_node, label="Ground Truth", color="steelblue", linewidth=1.5)

# Prediction — forecast window only
ax.plot(time_forecast, pred_node, label="Mock Prediction", color="tomato",
        linewidth=1.5, linestyle="--")

# Mark the burn-in / forecast boundary
ax.axvline(BURN_IN, color="grey", linestyle=":", linewidth=1, label="Burn-in ↔ Forecast")
ax.axvspan(0, BURN_IN, alpha=0.06, color="grey")
ax.text(BURN_IN / 2, ax.get_ylim()[1], "Burn-in", ha="center", va="top",
        fontsize=9, color="grey")

ax.set_xlabel("Time step")
ax.set_ylabel("Water Level")
ax.set_title(f"Autoregressive Dry-Run — {node_label}  (σ = {node_stds[node_idx]:.4f})")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"✅ Visualization complete for {node_label}.")